In [1]:
import re

with open('../the-verdict.txt', 'r', encoding='utf-8') as file:
    raw = file.read()

print(len(raw))
print(raw[0:100])

20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g


In [2]:
word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
word_array = [item.lower() for item in word_array if item.strip()]
print(word_array[0:20])
print(len(word_array))

['i', 'had', 'always', 'thought', 'jack', 'gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was']
4690


In [3]:
sorted_words = sorted(set(word_array))
sorted_words.extend(["<unk>", "<endoftext>"])
vocab_size = len(sorted_words)
print(vocab_size)

1078


In [4]:
vocab = {token:idx for idx,token in enumerate(sorted_words)}

for idx, (key,value) in enumerate(vocab.items()):
    print(key, value)
    if idx >= 20:
        break

! 0
" 1
' 2
( 3
) 4
, 5
-- 6
. 7
: 8
; 9
? 10
_ 11
a 12
abdication 13
able 14
about 15
above 16
abruptly 17
absolute 18
absorbed 19
absurdity 20


In [5]:
class Tokenizer():
    def __init__(self, vocab_dict):
        # Two dictionaries for changing the format of the data
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab_dict.items()}

    def encode(self, raw):

        # Return list of ids for any passed in raw text
        word_array = re.split(r'([_,.!-();:?"\']|--|\s)', raw)
        word_array = [item.lower() for item in word_array if item.strip()]

        # Handle unkept words
        word_array = [
            item if item in self.str_to_int
            else "<unk>" for item in word_array
        ]

        ids = [self.str_to_int[i] for i in word_array]
        return ids

    def decode(self, ids):
        # Convert list of ids back into text
        text = " ".join(self.int_to_str[i] for i in ids)
        text = re.sub(r'\s+([,.?!"\'])', r'\1', text)
        return text
        

In [6]:
# Lets run a quck demo on an instance of the tokenizer

tokenizer = Tokenizer(vocab)

text1 = "Hello, I am Neel!"
text2 = "How are you today?"
text = " <endoftext> ".join([text1, text2])
print(text)

Hello, I am Neel! <endoftext> How are you today?


In [7]:
tokenizer.encode(text)

[1076, 5, 481, 47, 1076, 0, 1077, 478, 66, 1072, 1076, 10]

In [8]:
tokenizer.decode(tokenizer.encode(text))

'<unk>, i am <unk>! <endoftext> how are you <unk>?'

In [9]:
import tiktoken
bpe_tokenizer = tiktoken.get_encoding('gpt2')

In [10]:
encoded_text = bpe_tokenizer.encode(raw)

In [11]:
# Let's create input-target pairs 

encoded_sample = encoded_text[50:]
context_size = 4
x = encoded_sample[:context_size]
y = encoded_sample[1:context_size+1] # Y is one more than x
print(x)
print(y)

[290, 4920, 2241, 287]
[4920, 2241, 287, 257]


In [12]:
import torch
from torch.utils.data import Dataset, DataLoader

class LLMDataset(Dataset):

    def __init__(self, text, tokenizer, length, stride):

        self.input_ids = []
        self.target_ids = []

        ids = tokenizer.encode(text)

        # Loop through data and append input and target tensors to the id arrays
        for i in range(0, len(ids) - length - 1, stride):

            input = ids[i:i+length]
            target = ids[i+1:i+1+length]

            self.input_ids.append(torch.tensor(input))
            self.target_ids.append(torch.tensor(target))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [13]:
def create_dataloader(text, batch_size=4, length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):

    tokenizer = tiktoken.get_encoding('gpt2')

    dataset = LLMDataset(text, tokenizer, length, stride)

    dataloader = DataLoader(
        dataset, batch_size, shuffle, drop_last=drop_last, num_workers=num_workers
    )

    return dataloader

In [14]:
vocab_size = 50257
output_dim = 256

embd = torch.nn.Embedding(vocab_size, output_dim)

context_length = 4
dataloader = create_dataloader(raw, 8, context_length, stride=context_length, shuffle=False)

In [15]:
data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [16]:
print(inputs)
print(inputs.shape)

tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])
torch.Size([8, 4])


In [17]:
token_embeddings = embd(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


In [18]:
positional_embd_layer = torch.nn.Embedding(context_length, output_dim)
positional_embd = positional_embd_layer(torch.arange(context_length))

In [19]:
positional_embd.shape

torch.Size([4, 256])

In [20]:
input_embd = token_embeddings + positional_embd # Uses broadcasting to convert positional embd shape
input_embd.shape

torch.Size([8, 4, 256])

In [21]:
import torch.nn as nn

class SelfAttention(nn.Module):

    def __init__(self, dim_in, dim_out):
        super().__init__()

        # Initialize the query, key, and value weights
        self.W_query = nn.Linear(dim_in, dim_out, bias=False)
        self.W_key = nn.Linear(dim_in, dim_out, bias=False)
        self.W_value = nn.Linear(dim_in, dim_out, bias=False)

    def forward(self, x):
        # x shape: (batch_size, seq_len, num_tokens)
        
        query = self.W_query(x)
        keys = self.W_key(x)
        value = self.W_value(x)

        # Calculate the dot product between the query and key vectors
        attention_scores = query @ keys.T
        attention_weights = torch.softmax(attention_scores / (keys.shape[-1] ** 0.5), dim = -1)
        context_vector = attention_weights @ value

        return context_vector

In [22]:
print(input_embd.shape)
sample_input = input_embd[0]

torch.Size([8, 4, 256])


In [23]:
selfattention = SelfAttention(256, 6)
selfattention(sample_input)

tensor([[ 0.1686,  0.2819, -0.4640,  0.2004, -0.4954,  0.6113],
        [ 0.0542,  0.4792, -0.5915,  0.4304, -0.3474,  0.0241],
        [ 0.0829,  0.6322, -0.5449,  0.3353, -0.8520,  0.0638],
        [ 0.0659,  0.4157, -0.5729,  0.3688, -0.2487,  0.3847]],
       grad_fn=<MmBackward0>)

In [24]:
# Trying to implement causal attention
class CasualSelfAttention(nn.Module):

    def __init__(self, dim_in, dim_out, context_length, dropout_val):
        super().__init__()

        self.dim_out = dim_out
        self.W_query = nn.Linear(dim_in, dim_out, bias=False)
        self.W_key = nn.Linear(dim_in, dim_out, bias=False)
        self.W_value = nn.Linear(dim_in, dim_out, bias=False)
        self.dropout = nn.Dropout(dropout_val)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        # x shape: (B, num tokens, vector emb size)
        batch, num_tokens, dim_in = x.shape

        queries = self.W_query(x) # B, num_tokens, d_out
        keys = self.W_key(x)
        values = self.W_value(x)
        # Take dot product between the query and keys
        # (B, num_tokens, d_out) x (B, d_out, num_tokens)
        attention_scores = queries @ keys.transpose(1,2) # (B, num_tokens, num_tokens)

        # Slice up self.mask to create causal mask for the correct num_tokens
        causal_mask = self.mask[:num_tokens, :num_tokens]
        attention_scores.masked_fill_(causal_mask == 1, -torch.inf)

        attention_weights = torch.softmax(attention_scores / (self.dim_out ** 0.5), dim=-1)
        attention_weights = self.dropout(attention_weights) # (B, num_tokens, dim_out)

        # Calculate context vector
        # (B, num_tokens, num_tokens) x (B, num_tokens, d_out)
        context_vector = attention_weights @ values # (B, num_tokens, d_out)
        return context_vector

In [25]:
input_embd = input_embd[0:2]

In [26]:
input_embd.shape


torch.Size([2, 4, 256])

In [27]:
causal_attention = CasualSelfAttention(256, 4, 4, 0.1)

In [28]:
print(causal_attention(input_embd))

tensor([[[-0.1334, -0.0398,  0.6757, -0.6979],
         [ 0.3118,  0.2878,  0.9331, -0.6451],
         [ 0.8888,  0.7217,  1.2365, -0.4000],
         [ 0.1078, -0.1148,  0.6387, -0.4008]],

        [[-0.2924, -0.4649,  0.7791,  0.6921],
         [-0.3030, -0.4407,  0.7590,  0.6613],
         [ 0.1795, -0.0606, -0.0049,  0.1014],
         [ 0.0423, -0.3397,  0.2761,  0.3393]]], grad_fn=<UnsafeViewBackward0>)


In [29]:
class MultiHeadAttentionWrapper(nn.Module):

    def __init__(self, num_heads, d_in, d_out, context_length, dropout_val):
        super().__init__()
        self.heads = [
            CasualSelfAttention(d_in, d_out, context_length, dropout_val)
            for _ in range(num_heads)
        ]

    def forward(self, x):

        return torch.cat([head(x) for head in self.heads], dim=-1)

In [30]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout_val, num_heads):

        super().__init__()

        # Ensure that the num_heads and d_out values are compatible
        assert(d_out % num_heads == 0)

        self.W_query = nn.Linear(d_in, d_out, bias=False)
        self.W_key = nn.Linear(d_in, d_out, bias=False)
        self.W_value = nn.Linear(d_in, d_out, bias=False)
        self.num_heads = num_heads
        self.d_out = d_out
        self.head_dim = d_out // num_heads
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))
        self.linear_layer = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout_val)

    def forward(self, x):

        # x: (B, num_tokens, d_in)
        batch, num_tokens, d_in = x.shape

        # (b, num_tokens, d_out)
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # Split into the heads
        keys = keys.view(batch, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(batch, num_tokens, self.num_heads, self.head_dim)
        values = values.view(batch, num_tokens, self.num_heads, self.head_dim)

        # Transpose so that shape is (batch, num_heads, num_tokens, head_dims)
        keys = keys.transpose(1,2)
        queries = queries.transpose(1,2)
        values = values.transpose(1,2)

        # Calculate the attention scores
        # (B, num_heads, num_tokens, head_dim) x (B, num_heads, head_dim, num_tokens)
        # (B, num_heads, num_tokens, num_tokens)
        attention_scores = keys @ queries.transpose(2,3) 

        # Create mask and apply
        causal_mask = self.mask[:num_tokens][:num_tokens]
        attention_scores.masked_fill_(causal_mask == 1, -torch.inf)

        attention_scores = attention_scores / (self.head_dim ** 0.5)
        attention_weights = torch.softmax(attention_scores, dim=-1) # Across
        attention_weights = self.dropout(attention_weights)
        # (B, num_heads, num_tokens, num_tokens) x (B, num_heads, num_tokens, head_dim)
        # (B, num_heads, num_tokens, head_dim)
        context_vector = attention_weights @ values

        # Reshape so that num_tokens is first
        context_vector = context_vector.transpose(1,2)
        context_vector = context_vector.contiguous().view(batch, num_tokens, self.d_out)
        context_vector = self.linear_layer(context_vector)

        return context_vector
        

In [31]:
input_embd.shape

torch.Size([2, 4, 256])

In [32]:
multiheadattn = MultiHeadAttention(256, 4, 4, 0.0, 2)

In [33]:
multiheadattn(input_embd)

tensor([[[ 0.8229,  0.4282, -0.4514,  0.1064],
         [ 0.4032,  0.4151, -0.2850, -0.0651],
         [ 0.2454,  0.7457, -0.0704,  0.1752],
         [ 0.2182,  0.6774, -0.0420,  0.2983]],

        [[ 0.6836,  0.2074, -0.3346, -0.0864],
         [ 0.2811,  0.4772, -0.2864, -0.0105],
         [ 0.0270,  0.6725, -0.3467, -0.0714],
         [ 0.1108,  0.4813, -0.4745, -0.0697]]], grad_fn=<ViewBackward0>)

In [34]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1 
}

In [ ]:
class FeedForward(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(config['emb_dim'], 4*config['emb_dim'])
            nn.GELU(approximate='tanh'),
            nn.Linear(4*config['emb_dim'], config['emb_dim'])
        )

    def forward(self, x):
        return self.layers(x)

In [ ]:
class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.attention = MultiHeadAttention(
            config['emb_dim'],
            config['emb_dim'],
            config['context_length'],
            config['drop_rate'],
            config['n_heads'],
        )
        self.forward_block = FeedForward(config)
        self.dropout = nn.Dropout(config['drop_rate'])
        self.layer_norm = nn.LayerNorm(config['embd_dim'])

    def forward(self, x):

        shortcut = x
        x = self.layer_norm(x)
        x = self.attention(x) # (b, # tokens, d_out)
        x = self.dropout(x)
        x = x + shortcut
 
        shortcut = x
        x = self.layer_norm(x)
        x = self.forward_block(x)
        x = self.dropout(x)
        x = x + shortcut

        return x

In [ ]:
class GPTModel(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.tok_emb = nn.Embedding(config['vocab_size'], config['emb_dim']])
        self.pos_emb = nn.Embedding(config['context_length'], config['emb_dim']])
        self.drop = nn.Dropout(config['drop_rate'])

        self.transformer_blocks = nn.Sequential(
            *[TransformerBlock(config) for _ in range(config['n_layers'])]
        )

        self.final_layer_norm = nn.LayerNorm(config['emb_dim'])
        self.out_head = nn.Linear(
            config['emb_dim'], config['vocab_size'], bias=False
        )

    def forward(self, x):
        # (batch, seq_len)
        batch_size, sequence_length = x.shape 
        tok_embeds = self.tok_emb(x)
        pos_embeds = self.pos_emb(torch.arange(sequence_length, device=x.device))
        x = tok_embeds + pos_embeds
        x = self.drop(x)
        x = self.transformer_blocks(x)
        x = self.final_layer_norm(x)
        logits = self.out_head(x)
        return logits
        